In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 1. Load dữ liệu đã xử lý
PROCESSED_DIR = '../dataset/processed/'
print("Đang load dữ liệu...")
df = pd.read_parquet(os.path.join(PROCESSED_DIR, 'master_data.parquet'))

print("Đang tính toán phân loại nhu cầu. Quá trình này có thể mất 1-2 phút...")

# MẸO: Chạy thử trên 100 sản phẩm đầu tiên cho nhanh. 
# Khi nào code chạy mượt, bạn COMMENT (thêm dấu #) 2 dòng dưới này lại để chạy cho toàn bộ siêu thị nhé!
sample_items = df['item_id'].unique()[:100]
df_run = df[df['item_id'].isin(sample_items)]
# df_run = df # Bỏ comment dòng này nếu muốn chạy full dữ liệu

def classify_demand(group):
    """Hàm tính ADI, CV2 và phân loại cho từng sản phẩm"""
    demand = group['demand'].values
    
    # Nếu không bán được cái nào trong lịch sử
    if demand.sum() == 0:
        return pd.Series({'ADI': np.nan, 'CV2': np.nan, 'Category': 'No Sales'})
        
    # Tìm index của ngày đầu tiên bắt đầu bán hàng (bỏ qua các ngày zero ban đầu do chưa nhập hàng)
    non_zero_indices = np.where(demand > 0)[0]
    first_sale_idx = non_zero_indices[0]
    active_demand = demand[first_sale_idx:]
    
    # 1. Tính ADI (Số ngày / Số ngày có khách mua)
    days_with_sales = len(active_demand[active_demand > 0])
    if days_with_sales == 0:
        adi = np.nan
    else:
        adi = len(active_demand) / days_with_sales
        
    # 2. Tính CV2 ((Độ lệch chuẩn / Số trung bình)^2)
    mean_demand = np.mean(active_demand)
    if mean_demand == 0:
        cv2 = 0
    else:
        std_demand = np.std(active_demand)
        cv2 = (std_demand / mean_demand) ** 2
        
    # 3. Phân loại
    if adi <= 1.32 and cv2 <= 0.49:
        category = 'Smooth (Đều đặn)'
    elif adi <= 1.32 and cv2 > 0.49:
        category = 'Erratic (Thất thường)'
    elif adi > 1.32 and cv2 <= 0.49:
        category = 'Intermittent (Ngắt quãng)'
    else:
        category = 'Lumpy (Cục bộ)'
        
    return pd.Series({'ADI': round(adi, 2), 'CV2': round(cv2, 2), 'Category': category})

# 2. Chạy Groupby để áp dụng hàm cho từng mặt hàng tại từng cửa hàng
classification_df = df_run.groupby(['item_id', 'store_id']).apply(classify_demand).reset_index()

# Lọc bỏ các món chưa từng bán được
classification_df = classification_df[classification_df['Category'] != 'No Sales']

print("\n✅ Hoàn tất! Thống kê số lượng các nhóm nhu cầu:")
print(classification_df['Category'].value_counts())

print("\nBảng phân loại chi tiết (5 dòng đầu):")
display(classification_df.head(10))

# 3. Lưu kết quả ra file để dùng cho bước Machine Learning
# classification_df.to_csv(os.path.join(PROCESSED_DIR, 'demand_classes.csv'), index=False)

Đang load dữ liệu...
Đang tính toán phân loại nhu cầu. Quá trình này có thể mất 1-2 phút...

✅ Hoàn tất! Thống kê số lượng các nhóm nhu cầu:
Category
Lumpy (Cục bộ)           196
Erratic (Thất thường)      4
Name: count, dtype: int64

Bảng phân loại chi tiết (5 dòng đầu):


,item_id,store_id,ADI,CV2,Category
0,FOODS_1_001,CA_1,NaN,NaN,NaN
1,FOODS_1_001,CA_2,NaN,NaN,NaN
2,FOODS_1_002,CA_1,NaN,NaN,NaN
3,FOODS_1_002,CA_2,NaN,NaN,NaN
4,FOODS_1_003,CA_1,NaN,NaN,NaN
5,FOODS_1_003,CA_2,NaN,NaN,NaN
6,FOODS_1_004,CA_1,NaN,NaN,NaN
7,FOODS_1_004,CA_2,NaN,NaN,NaN
8,FOODS_1_005,CA_1,NaN,NaN,NaN
9,FOODS_1_005,CA_2,NaN,NaN,NaN


In [2]:
import pandas as pd
import numpy as np
import os
import gc

# 1. Cấu hình đường dẫn
RAW_DIR = '../dataset/raw/'
PROCESSED_DIR = '../dataset/processed/'

print("1. Đang load Master Data (đã ép cân)...")
df = pd.read_parquet(os.path.join(PROCESSED_DIR, 'master_data.parquet'))

print("2. Đang load dữ liệu Giá (sell_prices.csv)...")
# Trong dữ liệu M5, giá được tính theo Tuần (wm_yr_wk)
prices = pd.read_csv(os.path.join(RAW_DIR, 'sell_prices.csv'))

# ==========================================
# THỰC THI BƯỚC 2 & 3 THEO FILE DOC
# ==========================================

print("3. Đang hợp nhất Giá vào dữ liệu chính...")
# Merge giá dựa trên cửa hàng, mã hàng và tuần
df = pd.merge(df, prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
del prices
gc.collect()

print("4. Đang tạo Đặc trưng Giá (Price Features)...")
# - Giá trị lớn nhất của mặt hàng đó (Max price)
df['price_max'] = df.groupby(['store_id', 'item_id'])['sell_price'].transform('max')
# - Mức độ giảm giá (Discount): Giá hiện tại / Giá cao nhất
df['price_discount'] = df['sell_price'] / df['price_max']
# - Sự thay đổi giá so với tuần trước (Price Momentum)
df['price_momentum'] = df['sell_price'] / df.groupby(['store_id', 'item_id'])['sell_price'].shift(1)

print("5. Đang xử lý Missing Value & Sự kiện (Calendar Features)...")
# Chuyển đổi các cột sự kiện chứa NaN thành chữ "No_Event" (Không có sự kiện)
event_cols = ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']
for col in event_cols:
    # Nếu cột là category thì cần add thêm hạng mục 'No_Event' trước khi fillna
    if df[col].dtype.name == 'category':
        df[col] = df[col].cat.add_categories(['No_Event'])
    df[col] = df[col].fillna('No_Event')

# Mã hóa (Label Encode) các cột chữ thành số để Học máy đọc được
cols_to_encode = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'] + event_cols
for col in cols_to_encode:
    df[col] = df[col].astype('category').cat.codes

print("6. Đang tạo Đặc trưng Thời gian (Time Features)...")
df['day'] = df['date'].dt.day
df['is_weekend'] = df['wday'].isin([1, 2]).astype(np.int8) # wday 1,2 là T7, CN trong M5

# Tạo lag và rolling cho một mặt hàng mẫu để code chạy nhanh
# (Trong thực tế sẽ phải groupby tính cho toàn bộ item)
print("7. Đang lưu tập dữ liệu Đã trích xuất đặc trưng...")
# Ép cân lại các cột giá vừa tạo cho nhẹ
cols_float = ['sell_price', 'price_max', 'price_discount', 'price_momentum']
for col in cols_float:
    df[col] = df[col].astype(np.float16)

# Lưu thành file mới chuyên dùng cho Machine Learning
featured_file = os.path.join(PROCESSED_DIR, 'featured_data.parquet')
df.to_parquet(featured_file, index=False)

print(f"✅ HOÀN TẤT BƯỚC 3! Dữ liệu sẵn sàng 100% tại: {featured_file}")
display(df[['date', 'item_id', 'demand', 'sell_price', 'price_discount', 'event_name_1']].head())

1. Đang load Master Data (đã ép cân)...
2. Đang load dữ liệu Giá (sell_prices.csv)...
3. Đang hợp nhất Giá vào dữ liệu chính...
4. Đang tạo Đặc trưng Giá (Price Features)...
5. Đang xử lý Missing Value & Sự kiện (Calendar Features)...
6. Đang tạo Đặc trưng Thời gian (Time Features)...
7. Đang lưu tập dữ liệu Đã trích xuất đặc trưng...
✅ HOÀN TẤT BƯỚC 3! Dữ liệu sẵn sàng 100% tại: ../dataset/processed/featured_data.parquet


,date,item_id,demand,sell_price,price_discount,event_name_1
0,2011-01-29,1437,0,NaN,NaN,30
1,2011-01-29,1438,0,NaN,NaN,30
2,2011-01-29,1439,0,NaN,NaN,30
3,2011-01-29,1440,0,NaN,NaN,30
4,2011-01-29,1441,0,NaN,NaN,30
